In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [8]:
data=pd.read_csv("spam_ham_dataset.csv")

In [9]:
data

,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\r\nthis deal is t...,0
...,...,...,...,...
5166,1518,ham,Subject: put the 10 on the ft\r\nthe transport...,0
5167,404,ham,Subject: 3 / 4 / 2000 and following noms\r\nhp...,0
5168,2933,ham,Subject: calpine daily gas nomination\r\n>\r\n...,0
5169,1409,ham,Subject: industrial worksheets for august 2000...,0


In [10]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5171 entries, 0 to 5170
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  5171 non-null   int64
 1   label       5171 non-null   str  
 2   text        5171 non-null   str  
 3   label_num   5171 non-null   int64
dtypes: int64(2), str(2)
memory usage: 5.3 MB


In [11]:
data.describe()

,Unnamed: 0,label_num
count,5171.000000,5171.000000
mean,2585.000000,0.289886
std,1492.883452,0.453753
min,0.000000,0.000000
25%,1292.500000,0.000000
50%,2585.000000,0.000000
75%,3877.500000,1.000000
max,5170.000000,1.000000


In [12]:
data.shape

(5171, 4)

In [14]:
data.shape[0]

5171

In [15]:
data.isnull().sum()

Unnamed: 0    0
label         0
text          0
label_num     0
dtype: int64

In [21]:
data.duplicated().sum()

0

In [23]:
data = data.drop('Unnamed: 0',axis=1)

In [24]:
data.head()

,label,text,label_num
0,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,spam,"Subject: photoshop , windows , office . cheap ...",1
4,ham,Subject: re : indian springs\r\nthis deal is t...,0


In [30]:
print(data['text'].iloc[0])

Subject: enron methanol ; meter # : 988291
this is a follow up to the note i gave you on monday , 4 / 3 / 00 { preliminary
flow data provided by daren } .
please override pop ' s daily volume { presently zero } to reflect daily
activity you can obtain from gas control .
this change is needed asap for economics purposes .


In [34]:
import re
import string
def clean_text(text):
    text = text.lower()
    text = re.sub(r'subject:','',text)
    text = re.sub(r'\d+','',text)
    text = text.translate(str.maketrans('','',string.punctuation))
    text = re.sub(r'\s+',' ',text).strip()
    return text

print(clean_text(data['text'].iloc[0]))

enron methanol meter this is a follow up to the note i gave you on monday preliminary flow data provided by daren please override pop s daily volume presently zero to reflect daily activity you can obtain from gas control this change is needed asap for economics purposes


In [35]:
data['clean_text'] = data['text'].apply(clean_text)
data[['text', 'clean_text']].head()

,text,clean_text
0,Subject: enron methanol ; meter # : 988291\r\n...,enron methanol meter this is a follow up to th...
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",hpl nom for january see attached file hplnol x...
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",neon retreat ho ho ho we re around to that mos...
3,"Subject: photoshop , windows , office . cheap ...",photoshop windows office cheap main trending a...
4,Subject: re : indian springs\r\nthis deal is t...,re indian springs this deal is to book the tec...


In [36]:
from sklearn.model_selection import train_test_split
X = data['clean_text']
y = data['label_num']

In [37]:
X_train, X_test, y_train, y_test, = train_test_split(X, y, test_size = 0.2, random_state=42, stratify=y)

In [38]:
print(X_train.shape, X_test.shape)

(4136,) (1035,)


In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf =  TfidfVectorizer(max_features=3000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print(X_train_tfidf.shape, X_test_tfidf.shape)

(4136, 3000) (1035, 3000)


In [42]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

In [44]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print("Accuracy:",accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test,y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test,y_pred))

Accuracy: 0.978743961352657

Classification report:
               precision    recall  f1-score   support

           0       0.99      0.98      0.98       735
           1       0.96      0.97      0.96       300

    accuracy                           0.98      1035
   macro avg       0.97      0.98      0.97      1035
weighted avg       0.98      0.98      0.98      1035


Confusion matrix:
 [[722  13]
 [  9 291]]


In [45]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.9420289855072463

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.94      0.96       735
           1       0.87      0.94      0.90       300

    accuracy                           0.94      1035
   macro avg       0.92      0.94      0.93      1035
weighted avg       0.94      0.94      0.94      1035


Confusion Matrix:
 [[692  43]
 [ 17 283]]


In [46]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_tfidf, y_train)

y_pred_rf = rf_model.predict(X_test_tfidf)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

Random Forest Accuracy: 0.9748792270531401

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.97      0.98       735
           1       0.94      0.98      0.96       300

    accuracy                           0.97      1035
   macro avg       0.96      0.98      0.97      1035
weighted avg       0.98      0.97      0.98      1035


Confusion Matrix:
 [[715  20]
 [  6 294]]


In [48]:
import joblib
joblib.dump(model, 'spam_classifier_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']